## Import necessary libraries

In [12]:
import sys
import torch
import torch_geometric
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import rdkit
import pandas as pd
import numpy as np
import scipy
import sklearn
import xgboost
import tqdm
import matplotlib
import seaborn
import streamlit
import requests
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
print("Success: All packages imported successfully!")
print(f"-> Using Python: {sys.executable}")
print(f"-> GPU Available: {torch.cuda.is_available()}")

Success: All packages imported successfully!
-> Using Python: /home/satyam-rana/miniconda3/envs/pytorch_env/bin/python
-> GPU Available: True


In [13]:
import os, json
import numpy as np

# Directory helpers
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_json(data, path: str):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(data, f)

def load_json(path: str):
    with open(path, "r") as f:
        return json.load(f)

# STITCH ID helpers 
def normalize_stitch_id(sid: str) -> str:
    """Normalize any STITCH ID to uppercase stripped string."""
    return str(sid).strip().upper()

def stitch_to_pubchem_cid(sid: str) -> int:
    """Extract numeric PubChem CID from a STITCH ID (CIDs/CID1 prefix)."""
    sid = normalize_stitch_id(sid)
    if sid.startswith("CID1"):
        return int(sid[4:])
    if sid.startswith("CIDS"):
        return int(sid[4:])
    numeric = "".join(c for c in sid if c.isdigit())
    return int(numeric) if numeric else -1

def compute_pos_weight(labels: np.ndarray) -> float:
    """Return neg/pos ratio for BCEWithLogitsLoss pos_weight."""
    pos = labels.sum()
    neg = len(labels) - pos
    return neg / max(pos, 1)

# Directory setup 
DATA_DIR      = "data"
RAW_DIR       = os.path.join(DATA_DIR, "raw")
PROC_DIR      = os.path.join(DATA_DIR, "processed")
MAP_DIR       = os.path.join(DATA_DIR, "mappings")
MODELS_DIR    = "models"

for d in [RAW_DIR, PROC_DIR, MAP_DIR, MODELS_DIR]:
    ensure_dir(d)

print("Directories ready:", DATA_DIR, MODELS_DIR)

Directories ready: data models


## Load dataset

In [14]:
def _load_csv(path, sep=","):
    df = pd.read_csv(path, sep=sep, low_memory=False)
    print(f"  {os.path.basename(path)}: shape={df.shape}, cols={list(df.columns)}")
    nulls = df.isnull().sum().sum()
    if nulls:
        print(f"    WARNING: {nulls} null values")
    print(df.head(3).to_string(), "\n")
    return df

def load_all_data(data_dir: str = RAW_DIR) -> dict:
    """Load and normalize all 9 dataset files. Returns dict of DataFrames."""
    print("=" * 60)
    print("Loading datasets from:", data_dir)
    print("=" * 60)
    data = {}

    # bio-decagon-combo (4.6M rows - load in chunks)
    chunks = []
    for chunk in pd.read_csv(os.path.join(data_dir, "bio-decagon-combo.csv"),
                             chunksize=100_000, low_memory=False):
        chunk.columns = [c.strip() for c in chunk.columns]
        for col in chunk.columns:
            if "stitch" in col.lower():
                chunk[col] = chunk[col].apply(normalize_stitch_id)
        chunks.append(chunk)
    data["combo"] = pd.concat(chunks, ignore_index=True)
    print(f"  bio-decagon-combo: shape={data['combo'].shape}")
    print(data["combo"].head(3).to_string(), "\n")

    for fname, key, sep in [
        ("bio-decagon-mono.csv",              "mono",          ","),
        ("bio-decagon-ppi.csv",               "ppi",           ","),
        ("bio-decagon-targets.csv",           "targets",       ","),
        ("bio-decagon-targets-all.csv",       "targets_all",   ","),
        ("bio-decagon-effectcategories.csv",  "effectcategories", ","),
        ("meddra_all_se.tsv",                 "meddra_se",     "\t"),
        ("meddra_freq.tsv",                   "meddra_freq",   "\t"),
    ]:
        data[key] = _load_csv(os.path.join(data_dir, fname), sep=sep)
        for col in data[key].columns:
            if "stitch" in col.lower():
                data[key][col] = data[key][col].apply(normalize_stitch_id)

    data["chembl"] = pd.read_csv(os.path.join(data_dir, "chembl_37_chemreps.txt"),
                                  sep="\t", low_memory=False)
    data["chembl"].columns = [c.strip() for c in data["chembl"].columns]
    print(f"  chembl_37_chemreps: shape={data['chembl'].shape}")
    print(data["chembl"].head(3).to_string(), "\n")

    print("All datasets loaded.")
    return data

# Run 
data = load_all_data(RAW_DIR)

Loading datasets from: data/raw
  bio-decagon-combo: shape=(4649441, 4)
       STITCH 1      STITCH 2 Polypharmacy Side Effect            Side Effect Name
0  CID000002173  CID000003345                 C0151714             hypermagnesemia
1  CID000002173  CID000003345                 C0035344  retinopathy of prematurity
2  CID000002173  CID000003345                 C0004144                 atelectasis 

  bio-decagon-mono.csv: shape=(174977, 3), cols=['STITCH', 'Individual Side Effect', 'Side Effect Name']
         STITCH Individual Side Effect              Side Effect Name
0  CID003062316               C1096328   central nervous system mass
1  CID003062316               C0162830     Photosensitivity reaction
2  CID003062316               C1611725  leukaemic infiltration brain 

  bio-decagon-ppi.csv: shape=(715612, 2), cols=['Gene 1', 'Gene 2']
   Gene 1  Gene 2
0  114787  375519
1  114787  285613
2  114787    7448 

  bio-decagon-targets.csv: shape=(18690, 2), cols=['STITCH', 'Gene']


In [15]:
def build_id_mappings(data: dict, save_dir: str = MAP_DIR) -> dict:
    """
    Build drug/protein/SE index dicts and save as JSON.
    CRITICAL: protein indices are offset by num_drugs - they never overlap with drug indices.
    """
    ensure_dir(save_dir)
    combo, mono, targets, targets_all, meddra_se = (
        data["combo"], data["mono"], data["targets"],
        data["targets_all"], data["meddra_se"])

    def stitch_cols(df):
        return [c for c in df.columns if "stitch" in c.lower()]

    # Collect all drug STITCH IDs from every file that has them
    drug_ids = set()
    for df in [combo, mono, targets, targets_all, meddra_se]:
        for col in stitch_cols(df):
            drug_ids.update(df[col].dropna().unique())
    drug_ids = sorted(drug_ids)
    
    # FIX: Cast drug IDs to string keys for JSON compliance
    drug_to_idx = {str(d): int(i) for i, d in enumerate(drug_ids)}
    idx_to_drug = {str(i): str(d) for d, i in drug_to_idx.items()}
    num_drugs = len(drug_ids)

    # Protein nodes - offset by num_drugs (CRITICAL: no overlap with drug indices)
    protein_ids = set()
    for df in [data["ppi"], targets, targets_all]:
        for col in df.columns:
            if "gene" in col.lower() or "protein" in col.lower():
                protein_ids.update(df[col].dropna().unique())
    protein_ids = sorted(protein_ids)
    
    # FIX: Cast protein IDs to string keys (This addresses the exact traceback crash)
    protein_to_idx = {str(p): int(i + num_drugs) for i, p in enumerate(protein_ids)}
    num_proteins = len(protein_ids)

    # Side effect indices
    combo_cols = list(combo.columns)
    se_id_col   = combo_cols[2]
    se_name_col = combo_cols[3]
    se_ids = sorted(combo[se_id_col].dropna().unique())
    
    # FIX: Cast side effect targets to clear primitives
    se_to_idx  = {str(s): int(i) for i, s in enumerate(se_ids)}
    se_to_name = {str(k): str(v) for k, v in zip(combo[se_id_col], combo[se_name_col]) if pd.notna(k)}

    # Drug pair index (key = sorted tuple stored as "A|B")
    stitch1_col, stitch2_col = combo_cols[0], combo_cols[1]
    pair_keys = (combo[[stitch1_col, stitch2_col]]
                 .drop_duplicates()
                 .apply(lambda r: f"{min(r[stitch1_col], r[stitch2_col])}|"
                                  f"{max(r[stitch1_col], r[stitch2_col])}", axis=1)
                 .drop_duplicates().tolist())
    drug_pair_to_idx = {str(k): int(i) for i, k in enumerate(pair_keys)}

    print(f"Drugs: {num_drugs}, Proteins: {num_proteins}, "
          f"Side effects: {len(se_ids)}, Drug pairs: {len(pair_keys)}")

    save_json(drug_to_idx,     os.path.join(save_dir, "drug_to_idx.json"))
    save_json(idx_to_drug,     os.path.join(save_dir, "idx_to_drug.json"))
    save_json(protein_to_idx,  os.path.join(save_dir, "protein_to_idx.json"))
    save_json(se_to_idx,       os.path.join(save_dir, "se_to_idx.json"))
    save_json(se_to_name,      os.path.join(save_dir, "se_to_name.json"))
    save_json(drug_pair_to_idx, os.path.join(save_dir, "drug_pair_to_idx.json"))

    return dict(drug_to_idx=drug_to_idx, idx_to_drug=idx_to_drug,
                protein_to_idx=protein_to_idx, se_to_idx=se_to_idx,
                se_to_name=se_to_name, drug_pair_to_idx=drug_pair_to_idx,
                num_drugs=num_drugs, num_proteins=num_proteins,
                combo_stitch1=stitch1_col, combo_stitch2=stitch2_col,
                combo_se_id=se_id_col, combo_se_name=se_name_col)

# Run 
mappings = build_id_mappings(data, MAP_DIR)

Drugs: 2135, Proteins: 19122, Side effects: 1317, Drug pairs: 63473


In [16]:
def build_splits(drug_pair_to_idx: dict, harm_labels: np.ndarray,
                 save_path: str = os.path.join(PROC_DIR, "splits.npz")):
    """80/10/10 stratified split saved as splits.npz."""
    ensure_dir(os.path.dirname(save_path))
    indices = np.arange(len(drug_pair_to_idx))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices, harm_labels, test_size=0.2, stratify=harm_labels, random_state=42)
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        pos = harm_labels[idx].sum()
        print(f"  {name}: {len(idx)} pairs — harmful={int(pos)} ({100*pos/len(idx):.1f}%)")

    np.savez(save_path, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
    print(f"Splits saved → {save_path}")
    return train_idx, val_idx, test_idx

In [17]:
import os
import sys
import time
import shutil
import requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

MORGAN_DIM = 2048
PUBCHEM_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IsomericSMILES/JSON"

# Define  target path in the CURRENT working directory
CWD_SAVE_PATH = os.path.join(os.getcwd(), "drug_features.npy")

# FALLBACK: Define where the file might have been saved previously
OLD_SAVE_PATH = os.path.join(PROC_DIR, "drug_features.npy") if 'PROC_DIR' in globals() else "./processed_data/drug_features.npy"

def _smiles_from_chembl(cid: int, chembl_df) -> str | None:
    inchi_col  = next((c for c in chembl_df.columns if "inchi" in c.lower() and "key" not in c.lower()), None)
    smiles_col = next((c for c in chembl_df.columns if "smiles" in c.lower()), None)
    if inchi_col is None or smiles_col is None:
        return None
    mask = chembl_df[inchi_col].astype(str).str.contains(str(cid), na=False)
    hits = chembl_df.loc[mask, smiles_col].dropna()
    return hits.iloc[0] if len(hits) > 0 else None

def _smiles_from_pubchem(cid: int, retry: int = 2) -> str | None:
    for _ in range(retry):
        try:
            r = requests.get(PUBCHEM_URL.format(cid=cid), timeout=5)
            if r.status_code == 200:
                return r.json()["PropertyTable"]["Properties"][0].get("IsomericSMILES")
        except Exception:
            time.sleep(0.5)
    return None

def _morgan_fp(smiles: str) -> np.ndarray:
    try:
        from rdkit import Chem
        from rdkit.Chem import rdFingerprintGenerator
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(MORGAN_DIM, dtype=np.float32)
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_DIM)
        fp = gen.GetFingerprintAsNumPy(mol)
        return fp.astype(np.float32)
    except Exception:
        return np.zeros(MORGAN_DIM, dtype=np.float32)

def build_drug_features(data: dict, drug_to_idx: dict, target_path: str = CWD_SAVE_PATH, fallback_path: str = OLD_SAVE_PATH) -> np.ndarray:
    # STEP 1: Check if it's already in the Current Working Directory 
    if os.path.exists(target_path):
        print(f"Found existing feature matrix: {target_path}")
        print("-> Loading pre-computed features")
        return np.load(target_path)
    
    # STEP 2: Check if it's in the old folder and copy it over 
    if os.path.exists(fallback_path):
        print(f"Found the pre-computed file tucked away at: {fallback_path}")
        print(f"-> Copying file to current working directory...")
        ensure_dir(os.path.dirname(target_path))
        shutil.copy2(fallback_path, target_path)
        print(f"Successfully copied to: {target_path}")
        return np.load(target_path)
        
    # --- STEP 3: Raw fallback (Only runs if the file literally doesn't exist anywhere) ---
    print(f"Pre-computed file not found anywhere. Starting generation...")
    chembl_df   = data["chembl"]
    mono_df     = data["mono"]
    meddra_se_df = data["meddra_se"]
    num_drugs   = len(drug_to_idx)

    # Build mono SE vocabulary
    mono_stitch_col  = next((c for c in mono_df.columns if "stitch" in c.lower()), mono_df.columns[0])
    mono_se_col      = mono_df.columns[1]
    meddra_stitch_col = next((c for c in meddra_se_df.columns if "stitch" in c.lower()), meddra_se_df.columns[0])
    meddra_se_col    = meddra_se_df.columns[-1]

    all_mono_ses = sorted(set(mono_df[mono_se_col].dropna()) | set(meddra_se_df[meddra_se_col].dropna()))
    mono_se_to_idx = {s: i for i, s in enumerate(all_mono_ses)}
    num_mono_se    = len(all_mono_ses)

    drug_mono = {d: set() for d in drug_to_idx}
    for _, row in mono_df.iterrows():
        if row[mono_stitch_col] in drug_mono:
            drug_mono[row[mono_stitch_col]].add(row[mono_se_col])
    for _, row in meddra_se_df.iterrows():
        if row[meddra_stitch_col] in drug_mono:
            drug_mono[row[meddra_stitch_col]].add(row[meddra_se_col])

    feature_dim   = MORGAN_DIM + num_mono_se
    drug_features = np.zeros((num_drugs, feature_dim), dtype=np.float32)

    def process_single_drug(item):
        drug_sid, idx = item
        cid = stitch_to_pubchem_cid(drug_sid)
        smiles = _smiles_from_chembl(cid, chembl_df)
        if smiles is None and cid > 0:
            smiles = _smiles_from_pubchem(cid)
        return idx, drug_sid, smiles

    print(f"Starting parallel processing with 20 workers for {num_drugs} drugs...")
    found_smiles = 0
    
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(process_single_drug, item) for item in drug_to_idx.items()]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Drug features"):
            idx, drug_sid, smiles = future.result()
            if smiles is not None:
                drug_features[idx, :MORGAN_DIM] = _morgan_fp(smiles)
                found_smiles += 1
            for se in drug_mono.get(drug_sid, set()):
                se_i = mono_se_to_idx.get(se)
                if se_i is not None:
                    drug_features[idx, MORGAN_DIM + se_i] = 1.0

    np.save(target_path, drug_features)
    print(f"File successfully baked onto disk at: {target_path}")
    return drug_features

# Run 
drug_features = build_drug_features(data, mappings["drug_to_idx"])

Found existing feature matrix: /home/satyam-rana/Desktop/deerhack 2026/drug_features.npy
-> Loading pre-computed features


In [18]:
import os
import torch
import pandas as pd

def _bidirectional(edge_index: torch.Tensor) -> torch.Tensor:
    """Add reverse edges so every edge (A,B) also has (B,A)."""
    if edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.cat([edge_index, edge_index.flip(0)], dim=1)

def build_graph(data: dict, drug_to_idx: dict, protein_to_idx: dict,
                save_dir: str = PROC_DIR) -> dict:
    """Build and save dd / dp / pp edge_index tensors using fast vectorization."""
    ensure_dir(save_dir)
    combo       = data["combo"]
    targets_all = pd.concat([data["targets"], data["targets_all"]], ignore_index=True).drop_duplicates()
    ppi         = data["ppi"]
    combo_cols  = list(combo.columns)

    # 1. Drug-drug edges (Fast Vectorized Map) 
    df_dd = combo[[combo_cols[0], combo_cols[1]]].drop_duplicates().copy()
    # Safely convert to string keys matching your JSON mapping dictionary
    s_col = df_dd[combo_cols[0]].astype(str).str.strip().map(drug_to_idx)
    d_col = df_dd[combo_cols[1]].astype(str).str.strip().map(drug_to_idx)
    
    valid_mask = s_col.notna() & d_col.notna()
    src = s_col[valid_mask].astype(int).tolist()
    dst = d_col[valid_mask].astype(int).tolist()
    dd_edge_index = _bidirectional(torch.tensor([src, dst], dtype=torch.long))
    torch.save(dd_edge_index, os.path.join(save_dir, "dd_edge_index.pt"))
    print(f"Drug-drug edges:     {dd_edge_index.shape[1]:>10,}")

    # 2. Drug-protein edges (Fast Vectorized Map) 
    drug_col = next((c for c in targets_all.columns if "stitch" in c.lower()), targets_all.columns[0])
    gene_col = next((c for c in targets_all.columns if "gene"   in c.lower()), targets_all.columns[1])
    
    df_dp = targets_all[[drug_col, gene_col]].copy()
    s_dp = df_dp[drug_col].astype(str).str.strip().map(drug_to_idx)
    d_dp = df_dp[gene_col].astype(str).str.strip().map(protein_to_idx)
    
    valid_dp_mask = s_dp.notna() & d_dp.notna()
    src_dp = s_dp[valid_dp_mask].astype(int).tolist()
    dst_dp = d_dp[valid_dp_mask].astype(int).tolist()
    dp_edge_index = _bidirectional(torch.tensor([src_dp, dst_dp], dtype=torch.long))
    torch.save(dp_edge_index, os.path.join(save_dir, "dp_edge_index.pt"))
    print(f"Drug-protein edges:  {dp_edge_index.shape[1]:>10,}")

    # 3. Protein-protein edges (Fast Vectorized Map) 
    gene_cols = [c for c in ppi.columns if "gene" in c.lower() or "protein" in c.lower()]
    
    df_pp = ppi[[gene_cols[0], gene_cols[1]]].copy()
    s_pp = df_pp[gene_cols[0]].astype(str).str.strip().map(protein_to_idx)
    d_pp = df_pp[gene_cols[1]].astype(str).str.strip().map(protein_to_idx)
    
    valid_pp_mask = s_pp.notna() & d_pp.notna()
    src_pp = s_pp[valid_pp_mask].astype(int).tolist()
    dst_pp = d_pp[valid_pp_mask].astype(int).tolist()
    pp_edge_index = _bidirectional(torch.tensor([src_pp, dst_pp], dtype=torch.long))
    torch.save(pp_edge_index, os.path.join(save_dir, "pp_edge_index.pt"))
    print(f"Protein-protein edges:{pp_edge_index.shape[1]:>9,}")

    # 4. Combined Tensor 
    full_edge_index = torch.cat([dd_edge_index, dp_edge_index, pp_edge_index], dim=1)
    torch.save(full_edge_index, os.path.join(save_dir, "full_edge_index.pt"))
    print(f"Total edges:         {full_edge_index.shape[1]:>10,}")

    return dict(dd=dd_edge_index, dp=dp_edge_index,
                pp=pp_edge_index, full=full_edge_index)

# Run 
edges = build_graph(data, mappings["drug_to_idx"], mappings["protein_to_idx"])

Drug-drug edges:        126,946
Drug-protein edges:     262,068
Protein-protein edges:1,431,224
Total edges:          1,820,238


In [19]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm

HARMFUL_CATEGORIES = {
    "cardiovascular system disease",
    "respiratory system disease",
    "thoracic disease",
    "gastrointestinal system disease",  # hepatobiliary approx.
    "urinary system disease",
    "hematopoietic system disease",
    "hematopoietic system diseases",
    "nervous system disease",
}

MIN_HARMFUL_SE = 20

def build_labels(data: dict, drug_pair_to_idx: dict, se_to_idx: dict,
                 save_dir: str = os.getcwd(),
                 min_harmful_se: int = MIN_HARMFUL_SE) -> tuple:
    if 'ensure_dir' in globals():
        ensure_dir(save_dir)

    combo      = data["combo"]
    eff        = data["effectcategories"]
    combo_cols = list(combo.columns)
    stitch1_col, stitch2_col, se_id_col = combo_cols[0], combo_cols[1], combo_cols[2]

    num_pairs = len(drug_pair_to_idx)
    num_se    = len(se_to_idx)

    print("Vectorizing side-effect pairings...")
    s1 = combo[stitch1_col].astype(str)
    s2 = combo[stitch2_col].astype(str)

    min_id    = np.where(s1 < s2, s1, s2)
    max_id    = np.where(s1 < s2, s2, s1)
    pair_keys = pd.Series(min_id) + "|" + pd.Series(max_id)

    mapped_pairs = pair_keys.map(drug_pair_to_idx)
    mapped_ses   = combo[se_id_col].astype(str).str.strip().map(se_to_idx)

    valid_mask = mapped_pairs.notna() & mapped_ses.notna()
    row_idx    = mapped_pairs[valid_mask].astype(int).values
    col_idx    = mapped_ses[valid_mask].astype(int).values
    data_vals  = np.ones(len(row_idx), dtype=np.float32)

    label_csr      = sp.csr_matrix((data_vals, (row_idx, col_idx)), shape=(num_pairs, num_se))
    label_csr.data = np.ones_like(label_csr.data)

    sp.save_npz(os.path.join(save_dir, "label_matrix.npz"), label_csr)
    density = label_csr.nnz / (num_pairs * num_se) if (num_pairs * num_se) > 0 else 0
    print(f"Label matrix shape: {label_csr.shape}, density: {density:.4f}")

    eff_id_col  = eff.columns[0]
    eff_cat_col = next((c for c in eff.columns
                        if "class" in c.lower() or "category" in c.lower()), eff.columns[-1])

    harmful_se_ids  = set(eff.loc[eff[eff_cat_col].isin(HARMFUL_CATEGORIES), eff_id_col]
                          .astype(str).str.strip().unique())
    harmful_cols    = np.array([int(se_to_idx[s]) for s in harmful_se_ids if s in se_to_idx])

    # Count distinct harmful SE per pair, then threshold
    cx           = label_csr.tocoo()
    harm_mask    = np.isin(cx.col, harmful_cols)
    harm_counts  = np.bincount(cx.row[harm_mask], minlength=num_pairs)
    harm_labels  = (harm_counts >= min_harmful_se).astype(np.float32)

    np.save(os.path.join(save_dir, "harm_labels.npy"), harm_labels)
    total = num_pairs
    hc    = int(harm_labels.sum())
    print(f"Threshold:         >= {min_harmful_se} harmful SE")
    print(f"Harmful pairs:     {hc} ({100*hc/total:.1f}%)" if total > 0 else "Harmful pairs: 0")
    print(f"Non-harmful pairs: {total-hc} ({100*(total-hc)/total:.1f}%)" if total > 0 else "Non-harmful pairs: 0")
    return label_csr, harm_labels

label_matrix, harm_labels = build_labels(
    data, mappings["drug_pair_to_idx"], mappings["se_to_idx"])

if 'build_splits' in globals():
    train_idx, val_idx, test_idx = build_splits(mappings["drug_pair_to_idx"], harm_labels)
else:
    print("Run split definition block before splitting data indices!")

Vectorizing side-effect pairings...
Label matrix shape: (63473, 1317), density: 0.0556
Threshold:         >= 20 harmful SE
Harmful pairs:     19173 (30.2%)
Non-harmful pairs: 44300 (69.8%)
  train: 50778 pairs — harmful=15338 (30.2%)
  val: 6347 pairs — harmful=1917 (30.2%)
  test: 6348 pairs — harmful=1918 (30.2%)
Splits saved → data/processed/splits.npz


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, BatchNorm as PyGBatchNorm


class DrugInteractionGNN(nn.Module):

    def __init__(
        self,
        drug_feature_dim: int,
        num_proteins: int,
        num_side_effects: int,
        protein_embed_dim: int = 256,
        hidden_dim: int = 256,
        embed_dim: int = 128,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.num_proteins = num_proteins
        self.embed_dim = embed_dim
        self.num_side_effects = num_side_effects

        # Node Embeddings and Projections
        self.protein_embedding = nn.Embedding(num_proteins, protein_embed_dim)
        self.drug_proj = nn.Linear(drug_feature_dim, hidden_dim)
        self.protein_proj = nn.Linear(protein_embed_dim, hidden_dim)

        # Graph Neural Network Layers
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.bn1 = PyGBatchNorm(hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, embed_dim)
        self.bn2 = PyGBatchNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

        # Diagonal bilinear decoder — one D_r per side effect
        self.diag_matrices = nn.Parameter(
            torch.empty(num_side_effects, embed_dim)
        )
        # FIX: Xavier uniform works perfectly on a 2D matrix directly without unsafe tensor views
        nn.init.xavier_uniform_(self.diag_matrices)

        # Prediction Head for Downstream Harmfulness Binary Classification
        self.harm_head = nn.Sequential(
            nn.Linear(num_side_effects, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def encode(
        self, drug_features: torch.Tensor, edge_index: torch.Tensor, num_drugs: int
    ) -> torch.Tensor:
        x_drug = F.relu(self.drug_proj(drug_features))
        prot_idx = torch.arange(self.num_proteins, device=drug_features.device)
        x_prot = F.relu(self.protein_proj(self.protein_embedding(prot_idx)))

        # Concatenate drug nodes [0...num_drugs-1] and protein nodes [num_drugs...]
        x = torch.cat([x_drug, x_prot], dim=0)

        # Graph convolutions with message passing
        x = self.dropout(F.relu(self.bn1(self.conv1(x, edge_index))))
        x = self.dropout(F.relu(self.bn2(self.conv2(x, edge_index))))
        return x  # [num_nodes, embed_dim]

    def decode_side_effects(
        self, z: torch.Tensor, drug_pairs: torch.Tensor
    ) -> torch.Tensor:
        """Vectorized diagonal bilinear decoder — returns [num_pairs, num_side_effects]."""
        z_i = z[drug_pairs[:, 0]]  # [P, E]
        z_j = z[drug_pairs[:, 1]]  # [P, E]

        # Compute dot product weighted by relation parameter weights for all relations at once
        # [P, 1, E] * [1, R, E] * [P, 1, E] -> sum over E -> [P, R]
        logits = (
            z_i.unsqueeze(1) * self.diag_matrices.unsqueeze(0) * z_j.unsqueeze(1)
        ).sum(dim=-1)
        return logits

    def decode_harmfulness(
        self, se_probs: torch.Tensor, freq_weights: torch.Tensor | None = None
    ) -> torch.Tensor:
        if freq_weights is not None:
            se_probs = se_probs * freq_weights.unsqueeze(0)
        return self.harm_head(se_probs)  # [P, 1]

    def forward(
        self,
        drug_features,
        edge_index,
        drug_pairs,
        num_drugs,
        freq_weights=None,
    ):
        z = self.encode(drug_features, edge_index, num_drugs)
        se_logits = self.decode_side_effects(z, drug_pairs)
        se_probs = torch.sigmoid(se_logits)
        harm_logits = self.decode_harmfulness(se_probs, freq_weights)
        return se_logits, harm_logits


# ── Run Sanity Check ───────────────────────────────────────────────────────
_dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {_dev}")

_num_se = len(mappings["se_to_idx"])
_num_drugs = mappings["num_drugs"]
_total_nodes = edges["full"].max().item() + 1
_num_proteins = _total_nodes - _num_drugs

_model = DrugInteractionGNN(
    drug_feature_dim=drug_features.shape[1],
    num_proteins=_num_proteins,
    num_side_effects=_num_se,
).to(_dev)

print(
    f"Parameters: {sum(p.numel() for p in _model.parameters() if p.requires_grad):,}"
)

Device: cuda
Parameters: 10,195,329


In [ ]:
import json, torch, numpy as np, scipy.sparse as sp

# Run this cell in Colab INSTEAD of the full pipeline above.
# Requires the zip to contain: data/processed/, data/mappings/
# Re-zip locally with:
#   zip -r gnn_colab_assets.zip data/processed/ data/mappings/

PROC = "data/processed"
MAP  = "data/mappings"

drug_features = np.load(f"{PROC}/drug_features.npy")

edges = {
    "dd":   torch.load(f"{PROC}/dd_edge_index.pt",   weights_only=True),
    "dp":   torch.load(f"{PROC}/dp_edge_index.pt",   weights_only=True),
    "pp":   torch.load(f"{PROC}/pp_edge_index.pt",   weights_only=True),
    "full": torch.load(f"{PROC}/full_edge_index.pt", weights_only=True),
}

harm_labels  = np.load(f"{PROC}/harm_labels.npy")
label_matrix = sp.load_npz(f"{PROC}/label_matrix.npz")

_splits   = np.load(f"{PROC}/splits.npz")
train_idx = _splits["train_idx"]
val_idx   = _splits["val_idx"]
test_idx  = _splits["test_idx"]

with open(f"{MAP}/drug_to_idx.json")      as f: drug_to_idx      = json.load(f)
with open(f"{MAP}/drug_pair_to_idx.json") as f: drug_pair_to_idx = json.load(f)
with open(f"{MAP}/se_to_idx.json")        as f: se_to_idx        = json.load(f)
with open(f"{MAP}/se_to_name.json")       as f: se_to_name       = json.load(f)
with open(f"{MAP}/protein_to_idx.json")   as f: protein_to_idx   = json.load(f)

num_drugs     = len(drug_to_idx)
_num_proteins = int(edges["full"].max().item()) + 1 - num_drugs

mappings = dict(
    drug_to_idx      = drug_to_idx,
    drug_pair_to_idx = drug_pair_to_idx,
    se_to_idx        = se_to_idx,
    se_to_name       = se_to_name,
    protein_to_idx   = protein_to_idx,
    num_drugs        = num_drugs,
    num_proteins     = _num_proteins,
)

print(f"drugs={num_drugs}, proteins={_num_proteins}, se={len(se_to_idx)}, pairs={len(drug_pair_to_idx)}")
print(f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
print("Assets loaded.")

## Model training loop

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score
import json as _json

TRAIN_CONFIG = dict(
    epochs=500,
    batch_size=512,
    lr=5e-4,
    weight_decay=1e-4,
    patience=30,
    se_loss_weight=0.6,
    harm_loss_weight=0.4,
)

def _build_freq_weights_safe(meddra_freq_df, se_to_idx: dict, device) -> torch.Tensor:
    weights = np.ones(len(se_to_idx), dtype=np.float32)
    if meddra_freq_df is not None:
        upper_col = next((c for c in meddra_freq_df.columns if "upper" in c.lower()), None)
        name_col  = meddra_freq_df.columns[-1]
        if upper_col:
            for _, row in meddra_freq_df.iterrows():
                idx = se_to_idx.get(str(row[name_col]))
                if idx is not None:
                    val = row[upper_col]
                    if isinstance(val, (int, float)) and not np.isnan(val):
                        weights[idx] = max(weights[idx], float(val))
    mx = weights.max()
    if mx > 0:
        weights /= mx
    return torch.tensor(weights, dtype=torch.float32, device=device)

def _build_pair_tensor(drug_pair_to_idx: dict, drug_to_idx: dict, device) -> torch.Tensor:
    pairs_sorted = sorted(drug_pair_to_idx.items(), key=lambda x: x[1])
    rows = []
    for key, _ in pairs_sorted:
        d1, d2 = key.split("|")
        rows.append([drug_to_idx.get(d1, 0), drug_to_idx.get(d2, 0)])
    return torch.tensor(rows, dtype=torch.long, device=device)

def train_model(cfg: dict = TRAIN_CONFIG):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    os.makedirs(MODELS_DIR, exist_ok=True)

    # ── Load assets ──────────────────────────────────────────────────────────
    drug_feat_t   = torch.tensor(drug_features, dtype=torch.float32, device=device)
    edge_index_t  = edges["full"].to(device)
    harm_labels_t = torch.tensor(harm_labels, dtype=torch.float32, device=device)

    _meddra      = data.get("meddra_freq") if isinstance(data, dict) else None
    freq_weights = _build_freq_weights_safe(_meddra, mappings["se_to_idx"], device)
    pair_tensor  = _build_pair_tensor(mappings["drug_pair_to_idx"], mappings["drug_to_idx"], device)

    num_drugs = mappings["num_drugs"]
    num_se    = len(mappings["se_to_idx"])

    # ── Class weights ────────────────────────────────────────────────────────
    se_train = label_matrix[train_idx]
    se_pos   = np.array(se_train.sum(axis=0)).flatten()
    se_neg   = len(train_idx) - se_pos
    se_pw    = torch.tensor(
        np.where(se_pos > 0, np.divide(se_neg, se_pos, where=se_pos > 0), 1.0),
        dtype=torch.float32, device=device)

    harm_pos = harm_labels[train_idx].sum()
    harm_pw  = torch.tensor(
        [len(train_idx) / max(harm_pos, 1) - 1],
        dtype=torch.float32, device=device)

    # Cache dense label matrix on CPU for fast batch slicing
    full_se_labels_cpu = torch.tensor(label_matrix.toarray(), dtype=torch.float32)

    # ── Model ────────────────────────────────────────────────────────────────
    model = DrugInteractionGNN(
        drug_feature_dim=drug_feat_t.shape[1],
        num_proteins=_num_proteins,
        num_side_effects=num_se,
    ).to(device)

    # Resume from checkpoint if it exists
    checkpoint = os.path.join(MODELS_DIR, "best_model.pt")
    if os.path.exists(checkpoint):
        model.load_state_dict(torch.load(checkpoint, map_location=device))
        print("✅ Resumed from checkpoint.")

    print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer      = Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler      = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8)
    se_criterion   = nn.BCEWithLogitsLoss(pos_weight=se_pw)
    harm_criterion = nn.BCEWithLogitsLoss(pos_weight=harm_pw)

    best_val_auc  = 0.0
    patience_left = cfg["patience"]

    for epoch in range(cfg["epochs"]):
        model.train()

        # ── Encode once per epoch, decoders trained per batch ────────────────
        with torch.no_grad():
            z_epoch = model.encode(drug_feat_t, edge_index_t, num_drugs)

        perm       = np.random.permutation(train_idx)
        epoch_loss = 0.0
        n_batches  = 0

        for start in range(0, len(perm), cfg["batch_size"]):
            batch_idx   = perm[start: start + cfg["batch_size"]]
            batch_pairs = pair_tensor[batch_idx]
            se_labels   = full_se_labels_cpu[batch_idx].to(device)
            harm_batch  = harm_labels_t[batch_idx].unsqueeze(1)

            optimizer.zero_grad()
            se_logits   = model.decode_side_effects(z_epoch, batch_pairs)
            se_probs    = torch.sigmoid(se_logits)
            harm_logits = model.decode_harmfulness(se_probs, freq_weights)

            loss = (cfg["se_loss_weight"]   * se_criterion(se_logits, se_labels) +
                    cfg["harm_loss_weight"] * harm_criterion(harm_logits, harm_batch))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        # ── Every 5 epochs: full forward pass to also train the encoder ───────
        if (epoch + 1) % 5 == 0:
            model.train()
            optimizer.zero_grad()
            perm_full        = np.random.permutation(train_idx)[:cfg["batch_size"] * 4]
            batch_pairs_full = pair_tensor[perm_full]
            se_labels_full   = full_se_labels_cpu[perm_full].to(device)
            harm_batch_full  = harm_labels_t[perm_full].unsqueeze(1)

            se_logits_full, harm_logits_full = model(
                drug_feat_t, edge_index_t, batch_pairs_full, num_drugs, freq_weights)
            loss_full = (cfg["se_loss_weight"]   * se_criterion(se_logits_full, se_labels_full) +
                         cfg["harm_loss_weight"] * harm_criterion(harm_logits_full, harm_batch_full))
            loss_full.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # ── Validation every 5 epochs ─────────────────────────────────────────
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                _, harm_v = model(drug_feat_t, edge_index_t,
                                  pair_tensor[val_idx], num_drugs, freq_weights)
                val_auc = roc_auc_score(harm_labels[val_idx],
                                        torch.sigmoid(harm_v).squeeze(1).cpu().numpy())

            scheduler.step(val_auc)
            print(f"Epoch {epoch+1:3d}/{cfg['epochs']} | loss={epoch_loss/n_batches:.4f} | val_auc={val_auc:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                torch.save(model.state_dict(), os.path.join(MODELS_DIR, "best_model.pt"))
                patience_left = cfg["patience"]
                print(f"  ✅ New best: {best_val_auc:.4f} — checkpoint saved.")
            else:
                patience_left -= 1
                if patience_left == 0:
                    print(f"Early stopping. Best val AUC: {best_val_auc:.4f}")
                    break

    return model

model = train_model()

In [23]:
import itertools

def predict_drug_pair(drug_ids: list):
    """
    drug_ids: list of STITCH IDs e.g. ['CID100000085', 'CID100000893']
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_drugs    = mappings["num_drugs"]
    freq_weights = _build_freq_weights_safe(None, mappings["se_to_idx"], device)
    drug_feat_t  = torch.tensor(drug_features, dtype=torch.float32, device=device)
    edge_index_t = edges["full"].to(device)

    # Load index mappings
    with open("data/mappings/se_to_name.json") as f:
        se_to_name = json.load(f)
    with open("data/mappings/se_to_idx.json") as f:
        se_to_idx = json.load(f)
    idx_to_se = {v: k for k, v in se_to_idx.items()}

    # Validate drugs exist
    for d in drug_ids:
        if d not in mappings["drug_to_idx"]:
            print(f"❌ Drug not found in mappings: {d}")
            print("Try one of these:", list(mappings["drug_to_idx"].keys())[:5])
            return

    # Build pair tensor
    pairs = list(itertools.combinations(drug_ids, 2))
    pair_indices = []
    for d1, d2 in pairs:
        i1 = mappings["drug_to_idx"][d1]
        i2 = mappings["drug_to_idx"][d2]
        pair_indices.append([i1, i2])

    pair_t = torch.tensor(pair_indices, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        z         = model.encode(drug_feat_t, edge_index_t, num_drugs)
        se_logits = model.decode_side_effects(z, pair_t)
        se_probs  = torch.sigmoid(se_logits).cpu().numpy()
        harm_logits = model.decode_harmfulness(
            torch.sigmoid(se_logits), freq_weights)
        harm_probs  = torch.sigmoid(harm_logits).squeeze(1).cpu().numpy()

    for i, (d1, d2) in enumerate(pairs):
        print(f"\n{'='*50}")
        print(f"Pair: {d1} + {d2}")
        print(f"{'='*50}")
        harmful  = harm_probs[i] >= 0.5
        risk     = "HIGH" if harm_probs[i] >= 0.7 else ("MODERATE" if harm_probs[i] >= 0.4 else "LOW")
        print(f"Harmful:    {'⚠️  YES' if harmful else '✅ NO'}")
        print(f"Confidence: {harm_probs[i]*100:.1f}%")
        print(f"Risk level: {risk}")

        # Top 10 predicted side effects
        top_idx = se_probs[i].argsort()[::-1][:10]
        print(f"\nTop 10 predicted side effects:")
        for rank, idx in enumerate(top_idx, 1):
            se_id   = idx_to_se.get(idx, str(idx))
            se_name = se_to_name.get(se_id, se_id)
            print(f"  {rank:2d}. {se_name:<40} ({se_probs[i][idx]*100:.1f}%)")

# ── Test with some drug IDs from your dataset ─────────────────────────────
# First, see what drugs are available
sample_drugs = list(mappings["drug_to_idx"].keys())[:10]
print("Sample drug IDs in your dataset:")
for d in sample_drugs:
    print(f"  {d}")

Sample drug IDs in your dataset:
  CID000000085
  CID000000119
  CID000000143
  CID000000158
  CID000000159
  CID000000187
  CID000000191
  CID000000206
  CID000000214
  CID000000222
